# SmolLM2-recurrent continued pretraining

Upload this notebook to Colab (Runtime > Change runtime type > GPU) and Run All. No editing required.

One-time setup, before the first run: open the key icon in the left sidebar (Secrets) and add:
- `HF_TOKEN` (required) -- a Hugging Face token with **write** access, from https://huggingface.co/settings/tokens. Used to pull/push checkpoints and the final model, so no Drive mount is needed.
- `WANDB_API_KEY` (optional) -- from https://wandb.ai/authorize. If missing, training still runs fine, just prints loss to the cell output instead of logging to wandb.

Everything else lives on the Hub, not Google Drive:
- Training data is re-mixed from FineWeb-Edu/DCLM/Cosmopedia-v2 into local (ephemeral) Colab disk each session -- it isn't persisted, so a fresh session re-downloads/re-mixes it.
- The **latest** full resumable checkpoint (weights + optimizer + dataloader state) is pushed to the private dataset repo `usr-wwelsh/smollm2-recurrent-checkpoints` every save, overwriting the previous one -- so it never accumulates and there's always exactly one copy to resume from, on the Hub or on local disk.
- Every time you reopen this notebook and Run All again, it checks that repo for a checkpoint and resumes from it instead of starting over. Free-tier Colab sessions time out well before the full run finishes, so you'll likely need to reopen and Run All several times.
- Once training finishes, the final weights are pushed to the public model repo `usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained`.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type and select a GPU, "
    "then Runtime > Restart session and run this cell again."
)

gpu_name = torch.cuda.get_device_name(0)
major, _minor = torch.cuda.get_device_capability(0)
no_amp = "false" if major >= 8 else "true"
print(f"GPU: {gpu_name} (compute capability {major}.{_minor}) -> no_amp={no_amp}")
print(
    "bf16 autocast enabled." if no_amp == "false" else
    "Pre-Ampere GPU (T4/P100) -- no native bf16, running in fp32 for correctness (slower per-step, but correct)."
)

In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

# Everything below is local Colab disk (ephemeral -- wiped when the session ends).
# Persistence across sessions comes entirely from the Hub repos, not this filesystem.
DATA_PATH = "/content/data/smollm2_recurrent_mix"
OUT_PATH = "/content/huginn_smollm2"
RUN_NAME = "smollm2-recurrent-v1"

CHECKPOINT_REPO = "usr-wwelsh/smollm2-recurrent-checkpoints"   # private HF dataset repo -- holds only the latest resumable checkpoint
FINAL_MODEL_REPO = "usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained"   # public HF model repo -- final trained weights land here

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(OUT_PATH, exist_ok=True)
print(f"Local data dir: {DATA_PATH}")
print(f"Local checkpoint dir: {OUT_PATH}")
print(f"Resume checkpoints from/to: {CHECKPOINT_REPO}")
print(f"Final model destination: {FINAL_MODEL_REPO}")

In [ ]:
import os

REPO_DIR = "/content/retrofitting-recurrence"
if not os.path.exists(REPO_DIR):
    !git clone --depth=1 https://github.com/usr-wwelsh/retrofitting-recurrence.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -q -r requirements.txt

In [ ]:
wandb_disabled = "true"
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb_disabled = "false"
    print("WANDB_API_KEY secret found -- wandb logging enabled.")
except Exception as e:
    print(f"No usable WANDB_API_KEY secret ({e}) -- wandb logging disabled, training still runs fine.")

In [ ]:
import glob

TOKEN_BUDGET = 500_000_000  # ~7,500 steps at batch_size=64 * max_length=1024 below

if glob.glob(f"{DATA_PATH}/*.parquet"):
    print(f"Found existing packed shards in {DATA_PATH}, skipping the mix step.")
else:
    print("No packed data found -- streaming and packing the FineWeb-Edu/DCLM/Cosmopedia-v2 mix.")
    print("This downloads/tokenizes ~500M tokens and can take a while; it resumes cleanly if interrupted (already-written shards are kept, just re-run this cell).")
    !python mix_smollm2_corpus.py --save_path="{DATA_PATH}" --token_budget={TOKEN_BUDGET}

In [ ]:
from huggingface_hub import hf_hub_download, HfApi

resume_flag = ""
api = HfApi()
if api.repo_exists(CHECKPOINT_REPO, repo_type="dataset"):
    resume_dir = f"{OUT_PATH}/resumed_checkpoint"
    hf_hub_download(repo_id=CHECKPOINT_REPO, repo_type="dataset", filename="chkpt.pt", local_dir=resume_dir)
    resume_flag = f"--resume_path={resume_dir}"
    print(f"Found a checkpoint on {CHECKPOINT_REPO} -- resuming from it.")
else:
    print(f"No checkpoint on {CHECKPOINT_REPO} yet -- starting a fresh run.")

In [ ]:
MAX_STEPS = 7500

!python train.py \
    --run_name={RUN_NAME} \
    --out_path={OUT_PATH} \
    --model_name=usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4 \
    --hub_checkpoint_repo={CHECKPOINT_REPO} \
    --preprocessed_data_path={DATA_PATH} \
    --is_parquet_dataset=true \
    --max_length=1024 \
    --micro_batch_size=8 \
    --batch_size=64 \
    --optim_config.lr=5e-5 \
    --scheduler_args.warmup=0.02 \
    --scheduler_args.cooldown=0.9 \
    --max_grad_norm=1.0 \
    --no_amp={no_amp} \
    --max_steps={MAX_STEPS} \
    --compile=false \
    --save_interval=250 \
    --wandb_disabled={wandb_disabled} \
    --mean_recurrence_schedule.turn_on=true \
    --mean_recurrence_schedule.warmup=0.25 \
    --mean_recurrence_schedule.max_mean_rec=4 \
    {resume_flag}

## Push final model to the Hub

Only finds something to push once training has actually reached `MAX_STEPS` in some session (checked via the checkpoint's saved step count, not just file presence) -- if the cell above got cut off by a session timeout, re-run this notebook (Run All) to resume training first.

In [ ]:
import re

ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
if not ckpt_dirs:
    print("No eval checkpoint found yet -- nothing to push.")
else:
    last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
    if last_step < MAX_STEPS:
        print(f"Latest checkpoint is at step {last_step:,}/{MAX_STEPS:,} -- training isn't finished yet, re-run this notebook to resume before pushing.")
    else:
        ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
        print(f"Training complete at step {last_step:,} -- pushing {ckpt_path} to {FINAL_MODEL_REPO}")
        api.create_repo(FINAL_MODEL_REPO, private=False, exist_ok=True)
        api.upload_folder(repo_id=FINAL_MODEL_REPO, folder_path=ckpt_path, commit_message=f"trained checkpoint @ step {last_step}")
        print("Done.")

## Eval

Sweeps recurrence depth on the latest local eval checkpoint and compares against base SmolLM2-360M, so you can see whether training is recovering toward (not stuck below) the base model. Works on any checkpoint reached so far -- doesn't require training to have finished.

In [ ]:
ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
if not ckpt_dirs:
    print("No model_only checkpoint found yet (training hasn't reached a save_interval boundary) -- nothing to eval.")
else:
    last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
    ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
    print(f"Evaluating {ckpt_path}")

    TASKS = "arc_easy,arc_challenge,hellaswag,mmlu,piqa,winogrande"
    for mean_recurrence in [1, 2, 4, 8]:
        out_dir = f"eval_outputs/{RUN_NAME}/step_{last_step}/mean_recurrence_{mean_recurrence}"
        !lm_eval --model hf \
            --model_args pretrained={ckpt_path},mean_recurrence={mean_recurrence},add_bos_token=True,dtype="float32",trust_remote_code=True \
            --tasks {TASKS} \
            --device cuda \
            --output_path {out_dir} \
            --batch_size auto

    !lm_eval --model hf \
        --model_args pretrained=HuggingFaceTB/SmolLM2-360M,add_bos_token=True,dtype="float32" \
        --tasks {TASKS} \
        --device cuda \
        --output_path eval_outputs/SmolLM2-360M-base \
        --batch_size auto